In [2]:
pip install mlxtend


   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 1.7 MB/s eta 0:00:01
   ------------------------------- -------- 1.0/1.4 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.9 MB/s  0:00:01


In [31]:
#importing libraries
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder


In [13]:
#Step 1: Create a dataframe and read the excel file
df = pd.read_excel("C:/Users/DISCOVERY/Big_Data_Minning_&_Analystics/Cassava_Yield_Data.xlsx")

In [32]:
df.head()

,Sesn,locn,block,rep,tillage,ferT,Plants_harvested,No_bigtubers,Weigh_bigtubers,No_mediumtubers,Weight_mediumtubers,No_smalltubers,Weight_smalltubers,Totaltuberno,AV_tubers_Plant,Total_tubweight,plotsize,HEC,TotalWeightperhectare,TotalTuberperHectare
0,2,1,1,1,conv,F2150,28,0,0.0,61,2.5,319,4.7,380,13.571429,7.2,5.3,10000,13584.905660,716981.132075
1,2,1,1,1,conv,F1100,28,0,0.0,110,4.6,260,4.0,370,13.214286,8.6,5.3,10000,16226.415094,698113.207547
2,2,1,1,1,conv,F3200,28,2,0.2,115,5.2,319,4.4,436,15.571429,9.8,5.3,10000,18490.566038,822641.509434
3,2,1,1,1,conv,F5300,28,6,0.7,60,2.7,303,4.8,369,13.178571,8.2,5.3,10000,15471.698113,696226.415094
4,2,1,1,1,conv,F4250,28,3,0.3,82,3.4,332,4.7,417,14.892857,8.4,5.3,10000,15849.056604,786792.452830


In [33]:
df.shape

(115, 20)

In [34]:
#check for missing data
df.isna().mean()

Sesn                     0.0
locn                     0.0
block                    0.0
rep                      0.0
tillage                  0.0
ferT                     0.0
Plants_harvested         0.0
No_bigtubers             0.0
Weigh_bigtubers          0.0
No_mediumtubers          0.0
Weight_mediumtubers      0.0
No_smalltubers           0.0
Weight_smalltubers       0.0
Totaltuberno             0.0
AV_tubers_Plant          0.0
Total_tubweight          0.0
plotsize                 0.0
HEC                      0.0
TotalWeightperhectare    0.0
TotalTuberperHectare     0.0
dtype: float64

In [35]:
# Step 2: Convert to transactions (fertiliser + season) as strings
transactions = df[['Sesn','ferT']].astype(str).values.tolist()
print(transactions)

[['2', 'F2150'], ['2', 'F1100'], ['2', 'F3200'], ['2', 'F5300'], ['2', 'F4250'], ['2', 'F5300'], ['2', 'F3200'], ['2', 'F4250'], ['2', 'F1100'], ['2', 'F2150'], ['2', 'F4250'], ['2', 'F5300'], ['2', 'F2150'], ['2', 'F3200'], ['2', 'F1100'], ['2', 'F2150'], ['2', 'F1100'], ['2', 'F3200'], ['2', 'F5300'], ['2', 'F4250'], ['2', 'F5300'], ['2', 'F3200'], ['2', 'F4250'], ['2', 'F1100'], ['2', 'F2150'], ['2', 'F4250'], ['2', 'F5300'], ['2', 'F2150'], ['2', 'F3200'], ['2', 'F1100'], ['1', 'F1100'], ['1', 'F3200'], ['1', 'F2150'], ['1', 'F4250'], ['1', 'F5300'], ['1', 'F1100'], ['1', 'F3200'], ['1', 'F2150'], ['1', 'F4250'], ['1', 'F5300'], ['1', 'F1100'], ['1', 'F3200'], ['1', 'F2150'], ['1', 'F4250'], ['1', 'F5300'], ['1', 'F1100'], ['1', 'F3200'], ['1', 'F2150'], ['1', 'F4250'], ['1', 'F5300'], ['1', 'F1100'], ['1', 'F3200'], ['1', 'F2150'], ['1', 'F4250'], ['1', 'F5300'], ['1', 'F1100'], ['1', 'F3200'], ['1', 'F2150'], ['1', 'F4250'], ['1', 'F5300'], ['1', 'F1100'], ['1', 'F3200'], ['1', '

In [26]:
# Step 3: One-hot encode
te = TransactionEncoder()
te_data = te.fit(transactions).transform(transactions)
df1 = pd.DataFrame(te_data, columns=te.columns_)


In [27]:
# Step 4: Frequent itemsets
frequent_itemsets = apriori(df1, min_support=0.05, use_colnames=True)

In [28]:
# Step 5: Association rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.1)

In [29]:
# Step 6: Unique fertilisers and seasons
ferT = df['ferT'].astype(str).unique().tolist()
Sesn = df['Sesn'].astype(str).unique().tolist()

In [30]:
# Step 7: Filter fertiliser ↔ season associations
fert_season_rules = rules[
    (rules['antecedents'].apply(lambda x: any(i in x for i in ferT)) &
     rules['consequents'].apply(lambda x: any(i in x for i in Sesn))) |
    (rules['antecedents'].apply(lambda x: any(i in x for i in Sesn)) &
     rules['consequents'].apply(lambda x: any(i in x for i in ferT)))
]

print(fert_season_rules[['antecedents','consequents','support','confidence','lift']])

   antecedents consequents   support  confidence  lift
0          (1)     (F1100)  0.095652    0.200000   1.0
1      (F1100)         (1)  0.095652    0.478261   1.0
2      (F2150)         (1)  0.095652    0.478261   1.0
3          (1)     (F2150)  0.095652    0.200000   1.0
4      (F3200)         (1)  0.095652    0.478261   1.0
5          (1)     (F3200)  0.095652    0.200000   1.0
6          (1)     (F4250)  0.095652    0.200000   1.0
7      (F4250)         (1)  0.095652    0.478261   1.0
8      (F5300)         (1)  0.095652    0.478261   1.0
9          (1)     (F5300)  0.095652    0.200000   1.0
10         (2)     (F1100)  0.104348    0.200000   1.0
11     (F1100)         (2)  0.104348    0.521739   1.0
12     (F2150)         (2)  0.104348    0.521739   1.0
13         (2)     (F2150)  0.104348    0.200000   1.0
14     (F3200)         (2)  0.104348    0.521739   1.0
15         (2)     (F3200)  0.104348    0.200000   1.0
16         (2)     (F4250)  0.104348    0.200000   1.0
17     (F4

Since the lift value is 1.0 in all the combinations, indicates **no meaningful association** between fertilizers and seasons in the dataset.